In [ ]:
!pip install  huggingface_hub
!pip install imagecorruptions
!pip install --upgrade scikit-image
# !pip install -U bitsandbytes>=0.46.1

In [ ]:
import os

# Environment path for imagecorruptions package
file_path = '/usr/local/lib/python3.12/dist-packages/imagecorruptions/corruptions.py'

if os.path.exists(file_path):
    # Read file contents
    with open(file_path, 'r') as file:
        file_data = file.read()

    # Patch deprecated parameter calls
    file_data = file_data.replace('multichannel=True', 'channel_axis=-1')
    file_data = file_data.replace('multichannel=False', 'channel_axis=None')

    # Overwrite patched file
    with open(file_path, 'w') as file:
        file.write(file_data)

    print("✅ Successfully patched corruptions.py via Python!")
else:
    print("❌ File not found. Please verify the file path.")

> **Note**: Execute environment setup cells above once upon fresh kernel initialization, then restart runtime if dependencies were updated.


In [ ]:
from datasets import load_dataset
import matplotlib.pyplot as plt
from pprint import pprint
import pickle
import numpy as np
np.float_ = np.float64

from pycocotools import mask as mask_util
from PIL import Image
import albumentations as A
import random
import torch
import os
import cv2

In [ ]:
def set_seed(seed=42):
    """
    Fix all random sources to ensure reproducible benchmark results.
    """
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # Multi-GPU support

    # Ensure deterministic cuDNN algorithms
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    print(f">>> Deterministic Seed set: {seed}")

# Initialize seed
set_seed(42)

In [ ]:
import os
from huggingface_hub import login

# Authenticate with Hugging Face Hub securely
# Priority: 1. Kaggle Secrets -> 2. Environment Variable -> 3. Interactive prompt
hf_token = None
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    pass

if not hf_token:
    hf_token = os.environ.get("HF_TOKEN")

if hf_token:
    login(token=hf_token)
    print("✅ Successfully logged in to Hugging Face Hub via secret/env token.")
else:
    login()  # Prompts securely in notebook without storing plain-text token


In [ ]:
import numpy as np
import random
from PIL import Image
from imagecorruptions import corrupt, get_corruption_names

class NoiseFactory:
    """
    Standard Benchmark Perturbation Factory (ImageNet-C):
    - Based on imagecorruptions library (15 standard types + 4 extra)
    - Input: PIL Image or Numpy Array
    - Output: PIL Image
    """

    # Supported corruptions from imagecorruptions library
    SUPPORTED_NOISES = get_corruption_names()

    @staticmethod
    def set_seed(seed=42):
        """Ensure deterministic execution across the entire benchmark."""
        random.seed(seed)
        np.random.seed(seed)

    @classmethod
    def apply(cls, image, noise_type, severity=1):
        """
        Core entry point.
        Accepts PIL or Numpy -> Converts to Numpy -> Calls corruptions library -> Returns PIL.
        """
        # 1. Verify corruption type validity
        if noise_type not in cls.SUPPORTED_NOISES:
            raise ValueError(f"Unsupported corruption type '{noise_type}'. "
                             f"Valid types include: {', '.join(cls.SUPPORTED_NOISES)}")

        # 2. Convert input (PIL / numpy -> uint8 RGB numpy array)
        if isinstance(image, Image.Image):
            # Ensure RGB color format before numpy conversion
            img_np = np.array(image.convert("RGB"))
        elif isinstance(image, np.ndarray):
            img_np = image.copy()
            # imagecorruptions expects uint8 input
            if img_np.dtype != np.uint8:
                img_np = img_np.astype(np.uint8)
        else:
            raise TypeError(f"Input must be a PIL Image or Numpy Array, received {type(image)}")

        # 3. Clamp severity level to range [1, 5]
        severity = max(1, min(severity, 5))

        # 4. Apply ImageNet-C benchmark corruption
        # imagecorruptions executes underlying perturbation
        noisy_np = corrupt(img_np, corruption_name=noise_type, severity=severity)

        # 5. Convert output (numpy -> PIL) and return
        return Image.fromarray(noisy_np)

    @classmethod
    def get_available_noises(cls):
        """Utility function to list all supported corruption types."""
        return cls.SUPPORTED_NOISES

In [ ]:
import cv2
import numpy as np
from PIL import Image

class AddNoise:
    def __init__(self, 
                 image_noise_type=None, image_severity=1, 
                 bbox_noise_type=None, bbox_severity=1, 
                 mask_noise_type=None, mask_severity=1):
        """
        Comprehensive 3-tier perturbation engine (Image - BBox - Mask) for benchmark evaluation.
        ...
        """
        self.image_noise_type = image_noise_type
        self.image_severity = image_severity

        self.bbox_noise_type = bbox_noise_type
        self.bbox_severity = bbox_severity

        self.mask_noise_type = mask_noise_type
        self.mask_severity = mask_severity

    # ==========================================
    # 1. IMAGE PERTURBATIONS (via NoiseFactory)
    # ==========================================
    def _apply_image_noise(self, image):
        if not self.image_noise_type:
            return image
            
        is_numpy = isinstance(image, np.ndarray)
        image_input = Image.fromarray(image) if is_numpy else image
            
        # Assumes NoiseFactory is defined
        noisy_pil = NoiseFactory.apply(
            image=image_input, 
            noise_type=self.image_noise_type, 
            severity=self.image_severity
        )

        return np.array(noisy_pil) if is_numpy else noisy_pil

    # ==========================================
    # 2. BOUNDING BOX SPATIAL PERTURBATIONS
    # ==========================================
    def xywh_to_xyxy(self, box):
        x, y, w, h = box
        return [x, y, x + w, y + h]

    # Extract bounding box [x, y, w, h] from binary mask
    def _extract_bbox_from_mask(self, mask):
        mask_np = np.array(mask)
        # Ensure 2D mask array
        if mask_np.ndim > 2:
            mask_np = mask_np[:, :, 0]
            
        y_indices, x_indices = np.where(mask_np > 0)
        
        # Handle empty mask edge case
        if len(y_indices) == 0 or len(x_indices) == 0:
            return [0, 0, 0, 0]
            
        xmin, xmax = np.min(x_indices), np.max(x_indices)
        ymin, ymax = np.min(y_indices), np.max(y_indices)
        
        w = xmax - xmin
        h = ymax - ymin
        
        return [int(xmin), int(ymin), int(w), int(h)]

    def _apply_bbox_noise(self, bboxes, W, H):
        if not getattr(self, 'bbox_noise_type', None):
            clean_bboxes = []
            for bbox in bboxes:
                x, y, w, h = bbox
                clean_bboxes.append(self.xywh_to_xyxy([int(x), int(y), int(w), int(h)]))
            return clean_bboxes          

        severity = max(1, min(getattr(self, 'bbox_severity', 1), 5))
        noisy_bboxes = []

        for bbox in bboxes:
            x, y, w, h = bbox
            cx, cy = x + w / 2, y + h / 2

            if self.bbox_noise_type == "shift":
                shift_ratio = severity * 0.05 
                dx = np.random.choice([-1, 1]) * (w * shift_ratio)
                dy = np.random.choice([-1, 1]) * (h * shift_ratio)
                cx, cy = cx + dx, cy + dy

            elif self.bbox_noise_type == "upscale":
                scale_ratio = 1.0 + (severity * 0.05)
                w, h = w * scale_ratio, h * scale_ratio

            elif self.bbox_noise_type == "downscale":
                scale_ratio = 1.0 - (severity * 0.05)
                w, h = w * scale_ratio, h * scale_ratio
                
            else:
                raise ValueError(f"Unsupported BBox perturbation: {self.bbox_noise_type}")

            nx = np.clip(cx - w / 2, 0, W - 1)
            ny = np.clip(cy - h / 2, 0, H - 1)
            nw = np.clip(w, 1, W - nx)
            nh = np.clip(h, 1, H - ny)         

            noisy_bboxes.append(self.xywh_to_xyxy([int(nx), int(ny), int(nw), int(nh)]))
            
        return noisy_bboxes

    
    # ==========================================
    # 3. MASK MORPHOLOGICAL PERTURBATIONS
    # ==========================================
    def _apply_mask_noise(self, mask, W, H):
        if not getattr(self, 'mask_noise_type', None):
            return mask
            
        mask_np = np.array(mask)
        is_binary_01 = mask_np.max() <= 1
        if is_binary_01:
            mask_np = (mask_np * 255).astype(np.uint8)
        else:
            mask_np = mask_np.astype(np.uint8)

        severity = max(1, min(getattr(self, 'mask_severity', 1), 5))
        
        if self.mask_noise_type in ["dilation", "erosion"]:
            # [SYNC FIX] Replaced harsh morphological kernels with Sub-pixel Rescaling
            # This perfectly matches the 5% BBox scaling logic and works on tiny objects.
            pos = np.where(mask_np > 0)
            
            # Safety: If mask is already empty, return it
            if pos[0].size == 0:
                noisy_mask = mask_np
            else:
                scale = 1.0 + (severity * 0.05) if self.mask_noise_type == "dilation" else 1.0 - (severity * 0.05)
                
                ymin, ymax = np.min(pos[0]), np.max(pos[0])
                xmin, xmax = np.min(pos[1]), np.max(pos[1])
                obj_w = xmax - xmin + 1
                obj_h = ymax - ymin + 1
                
                obj_crop = mask_np[ymin:ymax+1, xmin:xmax+1]
                
                # Calculate new size, ensuring it's at least 1x1 pixel
                new_w = max(1, int(round(obj_w * scale)))
                new_h = max(1, int(round(obj_h * scale)))
                
                rescaled_obj = cv2.resize(obj_crop, (new_w, new_h), interpolation=cv2.INTER_LINEAR)
                rescaled_obj = (rescaled_obj > 127).astype(np.uint8) * 255
                
                noisy_mask = np.zeros_like(mask_np)
                
                # Keep it centered based on the original bounding box
                start_y = ymin + (obj_h - new_h) // 2
                start_x = xmin + (obj_w - new_w) // 2
                
                # Safety Case: Ensure we don't paste outside image boundaries
                end_y = min(start_y + new_h, H)
                end_x = min(start_x + new_w, W)
                start_y = max(0, start_y)
                start_x = max(0, start_x)
                
                slice_h = end_y - start_y
                slice_w = end_x - start_x
                
                if slice_h > 0 and slice_w > 0:
                    noisy_mask[start_y:end_y, start_x:end_x] = rescaled_obj[:slice_h, :slice_w]
                else:
                    # Fallback if resizing pushed the object entirely off-screen
                    noisy_mask = mask_np

        elif self.mask_noise_type == "translation":
            pos = np.where(mask_np > 0)
            
            # Safety: If mask is already empty, return it
            if pos[0].size == 0:
                noisy_mask = mask_np
            else:
                # 1. Get the exact bounds of the mask
                ymin, ymax = np.min(pos[0]), np.max(pos[0])
                xmin, xmax = np.min(pos[1]), np.max(pos[1])
                obj_w = xmax - xmin + 1
                obj_h = ymax - ymin + 1
                
                # 2. Match the bbox severity ratio (0.05)
                shift_ratio = severity * 0.05
                tx = obj_w * shift_ratio
                ty = obj_h * shift_ratio
                
                # 3. Roll the dice for direction
                dx = np.random.choice([-1, 1]) * tx
                dy = np.random.choice([-1, 1]) * ty
                
                # 4. [EXACT SAFETY MATCH] 
                # Calculate the maximum allowed shift before the mask hits the image boundary.
                # We use np.clip on dx and dy to pin the mask to the edge, 
                # exactly mirroring how the BBox uses np.clip on its coordinates.
                max_dx_left = -xmin
                max_dx_right = (W - 1) - xmax
                max_dy_up = -ymin
                max_dy_down = (H - 1) - ymax
                
                dx = np.clip(dx, max_dx_left, max_dx_right)
                dy = np.clip(dy, max_dy_up, max_dy_down)
                
                # 5. Apply the safe shift
                M = np.float32([[1, 0, dx], [0, 1, dy]])
                noisy_mask = cv2.warpAffine(mask_np, M, (W, H))
            
        elif self.mask_noise_type == "boundary_jitter":
            contours, _ = cv2.findContours(mask_np, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            noisy_mask = np.zeros_like(mask_np)
            
            jitter_amp = severity * 2 
            
            jittered_contours = []
            for cnt in contours:
                noise = np.random.randint(-jitter_amp, jitter_amp + 1, cnt.shape)
                jittered_cnt = cnt + noise
                
                # [SAFETY FIX] Clip X and Y independently against W and H
                jittered_cnt[..., 0] = np.clip(jittered_cnt[..., 0], 0, W - 1)  # Clip X
                jittered_cnt[..., 1] = np.clip(jittered_cnt[..., 1], 0, H - 1)  # Clip Y
                
                jittered_contours.append(jittered_cnt)
                
            cv2.drawContours(noisy_mask, jittered_contours, -1, 255, thickness=cv2.FILLED)

        else:
            raise ValueError(f"Unsupported mask perturbation: {self.mask_noise_type}")

        if is_binary_01:
            noisy_mask = (noisy_mask > 127).astype(np.uint8)

        return Image.fromarray(noisy_mask) if isinstance(mask, Image.Image) else noisy_mask


    # ==========================================
    # UNIFIED DISPATCH ENTRY POINT
    # ==========================================
    def apply_to_data(self, image, bboxes=None, masks=None):
        if isinstance(image, np.ndarray):
            H, W = image.shape[:2]
        else:
            W, H = image.size 
            
        # 1. Apply image noise
        noisy_img = self._apply_image_noise(image)
        
        # Temporary holder for current bounding boxes
        current_bboxes = bboxes

        # 2. Apply mask noise and derive updated bounding boxes
        noisy_masks = None
        if masks is not None:
            if isinstance(masks, list):
                noisy_masks = [self._apply_mask_noise(m, W, H) for m in masks]
                # Re-derive bounding box from perturbed mask
                current_bboxes = [self._extract_bbox_from_mask(m) for m in noisy_masks]
            else:
                noisy_masks = self._apply_mask_noise(masks, W, H)
                # Re-derive bounding box from perturbed mask
                current_bboxes = [self._extract_bbox_from_mask(noisy_masks)]
        
        # 3. Apply spatial prompt noise to bounding boxes
        noisy_bboxes = self._apply_bbox_noise(current_bboxes, W, H) if current_bboxes else None

        return noisy_img, noisy_bboxes, noisy_masks

In [ ]:
import ast
import json
import os
import pickle
import numpy as np
from PIL import Image
from torch.utils.data import Dataset
from pycocotools import mask as mask_util

class UniversalDataset(Dataset):
    def __init__(self, hf_dataset, dataset_name, perturber=None, local_path=None):
        self.hf_dataset = hf_dataset
        self.dataset_name = dataset_name.lower()
        self.perturber = perturber # AddNoise perturber instance
        self.local_path = local_path

        # Dispatcher for dataset format parsers
        self.parsers = {
            'lvis': self._parse_standard,
            'cocostuff': self._parse_standard,
            'mapillary': self._parse_standard,
            'openimages': self._parse_standard, 
            'paco': self._parse_paco,
            'dlc_bench': self._parse_dlc_bench,
            'flickr30k': self._parse_flickr30k
        }

        if self.dataset_name not in self.parsers:
            raise ValueError(f"Dataset '{self.dataset_name}' is not supported.")

        # DLC-Bench is read from local storage, not Hugging Face
        if self.dataset_name == 'dlc_bench':
            self._init_dlc_bench()

    def _init_dlc_bench(self):
        """
        Read annotations.json from local disk,
        build img_dict, cat_dict, and a complete sample list with all fields.
        """
        if not self.local_path:
            raise ValueError("DLC-Bench requires local_path pointing to dataset directory.")

        anno_path    = os.path.join(self.local_path, "annotations.json")
        image_folder = os.path.join(self.local_path, "images")

        if not os.path.exists(anno_path):
            raise FileNotFoundError(f"File not found: {anno_path}")

        print(f"-> Loading annotations from: {anno_path}")
        with open(anno_path, 'r', encoding='utf-8') as f:
            coco_data = json.load(f)

        # Mapping: image ID -> image file name
        self._img_dict = {img['id']: img['file_name'] for img in coco_data.get('images', [])}

        # Mapping: category ID -> category name
        self._cat_dict = {cat['id']: cat['name'] for cat in coco_data.get('categories', [])}

        # Iterate through annotations and retain original fields
        self.dlc_samples = []
        for ann in coco_data.get('annotations', []):
            img_id = ann['image_id']
            if img_id not in self._img_dict:
                continue

            file_name     = self._img_dict[img_id]
            img_path      = os.path.join(image_folder, file_name)
            category_name = self._cat_dict.get(ann.get('category_id'), 'Unknown')

            self.dlc_samples.append({
                # Identity
                "ann_id":       ann['id'],
                "img_id":       img_id,
                "image_path":   img_path,
                # Semantics
                "category":     category_name,
                "caption":      category_name,
                # Localisation
                "bbox":         ann.get('bbox'),         # [x, y, w, h] COCO
                "segmentation": ann.get('segmentation'), # {"size":[H,W],"counts":str}
                "area":         ann.get('area', 0.0),
                # DLC-Bench exclusive flags
                "iscrowd":      int(ann.get('iscrowd',     0)),
                "isfake":       int(ann.get('isfake',      0)),
                "isreflected":  int(ann.get('isreflected', 0)),
                "frequency":    ann.get('frequency', ''), # F / C / R
                "size":         ann.get('size',      ''), # S / M / L (object size)
            })

        print(f"   [SUCCESS] Successfully generated {len(self.dlc_samples)} samples!")

    def __len__(self):
        if self.dataset_name == 'dlc_bench':
            return len(self.dlc_samples)
        return len(self.hf_dataset)

    def _get_empty_csv_row(self):
        """Flat 1D schema representing a single row in the exported CSV."""
        return {
            "dataset_name": self.dataset_name.upper(),
            "ann_id": "",
            "img_id": "",
            "image_path": "",    # Original image path for downstream loading
            "mask_size": "",     # Format: "H, W"
            "category": "",
            "caption": "",
            "mask_rle_counts": "", # RLE string
            "bboxes": "",        # Format: "[x, y, w, h]"
            "attributes": "",    # Comma-separated values
            "parts": "",         # Comma-separated values
            # Noise tracking parameters
            "image_noise_type": "",
            "image_severity": 0,
            "bbox_noise_type": "",
            "bbox_severity": 0,
            "mask_noise_type": "",
            "mask_severity": 0,
            # Placeholders for generated model captions
            "pred_keyword": "",
            "pred_phrase": "",
            "pred_detailed": "",
            # DLC-Bench exclusive fields (empty for standard datasets)
            "dlc_frequency":   "",  # F=frequent | C=common | R=rare
            "dlc_obj_size":    "",  # S=small | M=medium | L=large
            "dlc_iscrowd":     0,
            "dlc_isfake":      0,
            "dlc_isreflected": 0,
        }

    # ==========================================
    # Specialized parsing helpers (returning flattened rows)
    # ==========================================
    def _parse_standard(self, raw_ann):
        row = self._get_empty_csv_row()
        row["img_id"] = str(raw_ann.get('img_id', ''))
        row["ann_id"] = str(raw_ann.get('ann_id', ''))
        row["image_path"] = str(raw_ann.get('image', ''))
        row["category"] = str(raw_ann.get('category', ''))
        row["caption"] = str(raw_ann.get('caption', ''))
        
        # Decode RLE mask
        mask_rle = raw_ann.get('mask_rle')
        if mask_rle:
            row["mask_size"] = f"{mask_rle['size'][0]},{mask_rle['size'][1]}"
            counts = mask_rle.get('counts', '')
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts
            
        return row

    def _parse_paco(self, raw_ann):
        row = self._get_empty_csv_row()
        row["img_id"] = str(raw_ann.get('img_id', ''))
        row["ann_id"] = str(raw_ann.get('ann_id', ''))
        row["image_path"] = str(raw_ann.get('image', ''))
        
        row["category"] = str(raw_ann.get('category', raw_ann.get('object_name', '')))
        row["caption"] = str(raw_ann.get('caption', ''))
        
        # PACO contains attribute lists and object parts
        attrs = raw_ann.get('attributes', [])
        row["attributes"] = ", ".join(attrs) if isinstance(attrs, list) else str(attrs)
        
        part = raw_ann.get('part_name')
        row["parts"] = str(part) if part else ""

        # Prioritize object-level masks over generic masks
        mask_rle = raw_ann.get('object_mask_rle', raw_ann.get('mask_rle'))
        if mask_rle:
            row["mask_size"] = f"{mask_rle['size'][0]},{mask_rle['size'][1]}"
            counts = mask_rle.get('counts', '')
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts
            
        return row

    def _parse_dlc_bench(self, sample):
        """
        Accepts an entry from self.dlc_samples (pre-built from COCO JSON).
        Bypasses HF pickle — all fields are pre-resolved (img path, category name, etc.).

        Full schema of DLC-Bench (from diagnostic):
            ann_id, img_id, image_path  — identity
            category, caption           — semantics
            bbox    [x, y, w, h]        — COCO float coords
            segmentation {"size":[H,W], "counts":str}  — RLE mask
            area                        — pixel area
            iscrowd / isfake / isreflected — 0/1 flags
            frequency  F/C/R            — object frequency tier
            size       S/M/L            — object size tier
        """
        row = self._get_empty_csv_row()

        # Identity
        row["img_id"]     = str(sample["img_id"])
        row["ann_id"]     = str(sample["ann_id"])
        row["image_path"] = str(sample["image_path"])

        # Semantics
        row["category"] = sample["category"]
        row["caption"]  = sample["caption"]

        # BBox [x, y, w, h] as fallback; overridden by mask-derived bbox
        bbox = sample.get("bbox")
        if bbox and len(bbox) == 4:
            row["bboxes"] = str([round(float(v), 2) for v in bbox])

        # Segmentation RLE following COCO standard
        seg = sample.get("segmentation")
        if seg and isinstance(seg, dict) and 'size' in seg and 'counts' in seg:
            row["mask_size"] = f"{seg['size'][0]},{seg['size'][1]}"
            counts = seg['counts']
            if isinstance(counts, bytes):
                counts = counts.decode('utf-8')
            row["mask_rle_counts"] = counts

        # DLC-Bench exclusive flags
        row["dlc_frequency"]   = sample.get("frequency",   "")
        row["dlc_obj_size"]    = sample.get("size",        "")
        row["dlc_iscrowd"]     = sample.get("iscrowd",     0)
        row["dlc_isfake"]      = sample.get("isfake",      0)
        row["dlc_isreflected"] = sample.get("isreflected", 0)

        return row

    def _parse_flickr30k(self, raw_ann):
        return self._get_empty_csv_row()

    def xywh_to_xyxy(self, box):
        x, y, w, h = box
        return [x, y, x + w, y + h]

    # ==========================================
    # __getitem__: Load and apply perturbations
    # ==========================================
    def __getitem__(self, idx):

        # ── DLC-Bench: Load image from disk and parse annotations ──────
        if self.dataset_name == 'dlc_bench':
            sample = self.dlc_samples[idx]

            # Load image from disk
            try:
                image = Image.open(sample["image_path"]).convert("RGB")
            except Exception as e:
                print(f"   [ERROR] Failed to load image {sample['image_path']}: {e}")
                image = Image.new('RGB', (224, 224), (0, 0, 0))

            # Parse ra 1 row (DLC-Bench: 1 annotation = 1 sample)
            row = self._parse_dlc_bench(sample)
            raw_anns_parsed = [row]

        # ── Hugging Face datasets: Load from dataset object ───────────
        else:
            sample = self.hf_dataset[idx]

            # Flexible image key lookup ('image' or 'jpg')
            image = sample.get('jpg', sample.get('image'))
            if not isinstance(image, Image.Image):
                image = Image.fromarray(image)
            if image.mode != "RGB":
                image = image.convert("RGB")

            # 2. Read annotations list
            raw_anns = pickle.loads(sample['pickle'])
            if not isinstance(raw_anns, list):
                raw_anns = [raw_anns]

            parser_func = self.parsers[self.dataset_name]
            raw_anns_parsed = [parser_func(raw_ann) for raw_ann in raw_anns]

        rows_list = []
        binary_masks = []
        bboxes = []

        # 3. Parse sample metadata
        for row in raw_anns_parsed:
            
            # Decode RLE mask to binary mask for noise and bbox extraction
            b_mask = None
            if row["mask_rle_counts"] and row["mask_size"]:
                h, w = map(int, row["mask_size"].split(','))
                # Safe handling for string, bytes, and uncompressed list RLEs
                raw_counts = row["mask_rle_counts"]
                if isinstance(raw_counts, str):
                    valid_counts = raw_counts.encode('utf-8')
                else:
                    valid_counts = raw_counts # Keep list intact for pycocotools
                    
                rle_dict = {
                    'size': [h, w],
                    'counts': valid_counts 
                }
                b_mask = mask_util.decode(rle_dict)
            binary_masks.append(b_mask)

            # Derive bounding box from binary mask
            # Prioritize mask-derived bbox for cross-dataset consistency
            # Fallback to annotation bbox if mask is unavailable
            current_bbox = None
            if b_mask is not None:
                current_bbox = mask_util.toBbox(rle_dict).tolist()
            elif row["bboxes"]:
                # Fallback: use parsed bbox
                try:
                    current_bbox = ast.literal_eval(row["bboxes"])
                except (ValueError, SyntaxError):
                    current_bbox = None
            bboxes.append(current_bbox)
            
            rows_list.append(row)

        # 4. Apply configured perturbations
        if self.perturber:
            noisy_img, noisy_bboxes, noisy_masks = self.perturber.apply_to_data(
                image=image, 
                bboxes=bboxes, 
                masks=binary_masks
            )
            image = noisy_img
        else:
            noisy_bboxes = []
            for bbox in bboxes:
                if bbox is not None:
                    noisy_bboxes.append(self.xywh_to_xyxy(bbox))
                else:
                    noisy_bboxes.append(None)
            noisy_masks = binary_masks

        # 5. Update perturbed BBox, RLE mask, and perturbation metadata
        for i, row in enumerate(rows_list):
            # Store bbox coordinates as string "[x, y, w, h]"
            if noisy_bboxes is not None and noisy_bboxes[i] is not None:
                row["bboxes"] = str([round(x, 2) for x in noisy_bboxes[i]])
            
            n_mask = noisy_masks[i] if noisy_masks else None
            
            # Re-encode perturbed mask to RLE string
            if n_mask is not None:
                noisy_mask_f = np.asfortranarray(n_mask).astype(np.uint8)
                encoded_noisy_mask = mask_util.encode(noisy_mask_f)
                
                counts = encoded_noisy_mask['counts']
                if isinstance(counts, bytes):
                    counts = counts.decode('utf-8')
                row["mask_rle_counts"] = counts
                
                # Update mask dimensions
                row["mask_size"] = f"{encoded_noisy_mask['size'][0]},{encoded_noisy_mask['size'][1]}"
            
            # Record perturbation configuration
            if self.perturber:
                row["image_noise_type"] = str(self.perturber.image_noise_type)
                row["image_severity"] = self.perturber.image_severity
                row["bbox_noise_type"] = str(self.perturber.bbox_noise_type)
                row["bbox_severity"] = self.perturber.bbox_severity
                row["mask_noise_type"] = str(self.perturber.mask_noise_type)
                row["mask_severity"] = self.perturber.mask_severity

        return {
            "image": image,           # Transformed image for model inference
            "masks": noisy_masks,     # Target binary masks 
            "bboxes": noisy_bboxes,   # Target bounding boxes
            "csv_rows": rows_list     # Metadata rows for CSV export
        }

In [ ]:
# LVIS
# hf_dataset = load_dataset("nvidia/describe-anything-dataset", "LVIS")['train']
# print(len(hf_dataset))

# COCOStuff
# hf_dataset = load_dataset("nvidia/describe-anything-dataset", "COCOStuff")['train']
hf_dataset = load_dataset("Khoale11hcmut/gpt4o_captions_1k5samples")['train']
print(len(hf_dataset))

# PACO
# hf_dataset = load_dataset("nvidia/describe-anything-dataset", "PACO")['train']
# print(len(hf_dataset))


# Note: DLC-Bench is loaded locally from disk; no Hugging Face download required.
# hf_dataset = 1

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
import copy
import math

def visualize_benchmark_matrix(clean_dataset, start_index, end_index):
    """
    Render 110-test benchmark evaluation matrix for samples in range [start_index, end_index).
    Requirement: clean_dataset must be initialized with perturber=None.
    """
    print(f"\n[VISUALIZATION] Rendering 110-test benchmark matrix for images {start_index} to {end_index-1}...")

    # ==========================================
    # 1. DEFINE 110 BENCHMARK MATRIX TEST SUITES
    # ==========================================
    configs = []

    # a. Clean Baselines (Mask & BBox)
    configs.append({"title": "1. Base (GT Mask)", "params": {}, "mode": "mask"})
    configs.append({"title": "2. Base (GT BBox)", "params": {}, "mode": "bbox"})

    # b. Image Corruptions (15 types x 5 severities)
    img_noises = ['gaussian_noise', 'shot_noise', 'impulse_noise',
                  'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur',
                  'snow', 'frost', 'fog', 'brightness',
                  'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression']
    for n_type in img_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"Img: {n_type} (Lv{s})",
                "params": {"image_noise_type": n_type, "image_severity": s},
                "mode": "mask"
            })

    # c. Spatial BBox Noise (3 types x 5 severities)
    bbox_noises = ["shift", "upscale", "downscale"]
    for n_type in bbox_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"BBox: {n_type} (Lv{s})",
                "params": {"bbox_noise_type": n_type, "bbox_severity": s},
                "mode": "bbox"
            })

    # d. Mask Perturbations (4 types x 5 severities)
    mask_noises = ["dilation", "erosion", "translation", "boundary_jitter"]
    for n_type in mask_noises:
        for s in range(1, 6):
            configs.append({
                "title": f"Mask: {n_type} (Lv{s})",
                "params": {"mask_noise_type": n_type, "mask_severity": s},
                "mode": "mask"
            })

    # ==========================================
    # 2. RENDER MATRIX FOR EACH SAMPLE
    # ==========================================
    actual_end = min(end_index, len(clean_dataset))
    for idx in range(start_index, actual_end):
        print(f"-> Rendering sample index: {idx}...")

        # Extract clean sample from dataset
        batch = clean_dataset[idx]
        base_img = np.array(batch['image'])
        base_bboxes = batch['bboxes']
        base_masks = batch['masks']

        # Dynamic subplot grid: 5 columns
        n_cols = 5
        n_rows = math.ceil(len(configs) / n_cols)
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 5, n_rows * 5))
        axes = axes.flatten()
        fig.suptitle(f"Robustness Benchmark Matrix (110 Tests) - Sample: {idx}", fontsize=28, fontweight='bold', y=0.98)

        for i, cfg in enumerate(configs):
            ax = axes[i]
            params = cfg["params"]
            mode = cfg["mode"]

            # Instantiate AddNoise for current test
            perturber = AddNoise(**params) if params else None

            # Deepcopy sample data to isolate test conditions
            img_c = base_img.copy()
            bboxes_c = copy.deepcopy(base_bboxes) if base_bboxes else None
            masks_c = [m.copy() if m is not None else None for m in base_masks] if base_masks else None
            # print(bboxes_c)
            # Apply perturbation
            if perturber:
                new_bboxes_c = []
                for bbox in bboxes_c:
                    new_bboxes_c.append([bbox[0], bbox[1], bbox[2] - bbox[0], bbox[3] - bbox[1]])
                noisy_img, noisy_bboxes, noisy_masks = perturber.apply_to_data(
                    image=img_c, bboxes=new_bboxes_c, masks=masks_c
                )
            else:
                noisy_img, noisy_bboxes, noisy_masks = img_c, bboxes_c, masks_c

            # Render image
            if not isinstance(noisy_img, np.ndarray):
                noisy_img = np.array(noisy_img)
            ax.imshow(noisy_img)

            # Render region overlay (Mask or BBox)
            if mode == "mask" and noisy_masks:
                for mask in noisy_masks:
                    if mask is None:
                        continue
                    m_np = np.array(mask)
                    # Green translucent mask overlay
                    color_mask = np.zeros((*m_np.shape, 4))
                    color_mask[m_np > 0] = [0, 1, 0, 0.45]
                    ax.imshow(color_mask)

            elif mode == "bbox" and noisy_bboxes:
                for bbox in noisy_bboxes:
                    if bbox is None:
                        continue

                    # 1. Input format: [xmin, ymin, xmax, ymax]
                    x1, y1, x2, y2 = bbox

                    # 2. Compute width (w) and height (h) for matplotlib
                    w = x2 - x1
                    h = y2 - y1

                    # 3. Draw bounding box rectangle
                    rect = patches.Rectangle((x1, y1), w, h, linewidth=2, edgecolor='red', facecolor='none')
                    ax.add_patch(rect)

            # Set title and border styling
            ax.set_title(cfg["title"], fontsize=12, pad=6)
            ax.axis('off')

        # Turn off unused subplot axes
        for j in range(len(configs), len(axes)):
            axes[j].axis('off')

        plt.subplots_adjust(wspace=0.05, hspace=0.2, top=0.97)
        plt.show()



# visualize_benchmark_matrix(UniversalDataset(hf_dataset=hf_dataset, dataset_name="dlc_bench", perturber=None, local_path="/kaggle/input/datasets/ckhangnguyn/dlc-bench/DLC-Bench"), 0, 2)

In [ ]:
# =============================================================================
# shikra_pipeline.py
# Shikra-7B inference pipeline — bbox input → description or keyword output.
# =============================================================================

import gc
import re
from io import BytesIO
from typing import List, Optional, Tuple, Union

import requests
import torch
import torch.nn as nn
from PIL import Image
from torch.nn import CrossEntropyLoss
from transformers import (
    AutoTokenizer,
    LlamaTokenizer,
    CLIPImageProcessor,
    CLIPVisionConfig,
    CLIPVisionModel,
    LlamaConfig,
    LlamaForCausalLM,
    LlamaModel,
)
from transformers import logging as hf_logging
from transformers.cache_utils import Cache
from transformers.modeling_outputs import BaseModelOutputWithPast, CausalLMOutputWithPast


# =============================================================================
# CONSTANTS
# =============================================================================

DEFAULT_IMAGE_PATCH_TOKEN = "<im_patch>"
DEFAULT_IM_START_TOKEN    = "<im_start>"
DEFAULT_IM_END_TOKEN      = "<im_end>"

DEFAULT_IMAGE_TOKEN_LEN = 256
USE_IM_START_END        = True
MAX_NEW_TOKENS          = 512
MAX_NEW_TOKENS_QA       = 32       # keywords are short — no need for 512
TEMPERATURE             = 0.2
TOP_P                   = 0.7
DEVICE                  = "cuda" if torch.cuda.is_available() else "cpu"
CLIP_MODEL_NAME         = "openai/clip-vit-large-patch14"

SYSTEM_MSG = (
    "A chat between a curious user and an artificial intelligence assistant. "
    "The assistant gives helpful, detailed, and polite answers to the user's questions."
)


# PointQA question templates — bbox placeholder {bbox} is replaced at call time.
POINTQA_TEMPLATES = {
    "label":  "What is in region {bbox}?",
    "color":  "What color is the object in region {bbox}?",
    "count":  "How many objects are in region {bbox}?",
    "action": "What is the object doing in region {bbox}?",
    "describe_roi": "Describe the region {bbox} in detail",
}

GC_DETAILED_TEMPLATE = (
    "Give me a comprehensive description of the specified area <boxes> in the picture <image>."
)


# =============================================================================
# SUPPRESS NOISY WARNINGS
# =============================================================================

def suppress_load_warnings():
    hf_logging.set_verbosity_error()
    import urllib3
    urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)


# =============================================================================
# MODEL DEFINITION
# =============================================================================

class ShikraConfig(LlamaConfig):
    model_type = "shikra"


class ShikraLlamaModel(LlamaModel):
    config_class = ShikraConfig

    def __init__(self, config: LlamaConfig, mm_vision_tower=None, mm_hidden_size=None):
        super().__init__(config)
        if hasattr(config, "mm_vision_tower"):
            vision_config = CLIPVisionConfig.from_pretrained(config.mm_vision_tower)
            self.vision_tower = [CLIPVisionModel(vision_config)]
        if hasattr(config, "use_mm_proj"):
            self.mm_projector = nn.Linear(config.mm_hidden_size, config.hidden_size)

    def initialize_vision_modules(self, vision_tower, mm_vision_select_layer,
                                   pretrain_mm_mlp_adapter=None, tune_mm_mlp_adapter=False):
        self.config.mm_vision_tower = vision_tower
        image_processor = CLIPImageProcessor.from_pretrained(vision_tower)
        loaded_tower = CLIPVisionModel.from_pretrained(vision_tower)
        loaded_tower.requires_grad_(False)
        loaded_tower = loaded_tower.to(torch.float16)
        self.vision_tower = [loaded_tower]
        vision_config = loaded_tower.config
        num_patches = (vision_config.image_size // vision_config.patch_size) ** 2
        self.config.use_mm_proj            = True
        self.config.mm_hidden_size         = vision_config.hidden_size
        self.config.mm_vision_select_layer = mm_vision_select_layer
        if not hasattr(self, "mm_projector"):
            self.mm_projector = nn.Linear(vision_config.hidden_size, self.config.hidden_size)
        if pretrain_mm_mlp_adapter is not None:
            mm_projector_weights = torch.load(pretrain_mm_mlp_adapter, map_location="cpu")
            self.mm_projector.load_state_dict(
                {k.split(".")[-1]: v for k, v in mm_projector_weights.items()}
            )
        return dict(image_processor=image_processor, image_token_len=num_patches,
                    vision_config=vision_config)

    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, use_cache=None,
                output_attentions=None, output_hidden_states=None, images=None,
                return_dict=None, cache_position=None):
        orig_embeds_params = getattr(self, "orig_embeds_params", None)
        if inputs_embeds is None:
            inputs_embeds = self.embed_tokens(input_ids)
        vision_tower = getattr(self, "vision_tower", None)
        if (vision_tower is not None
                and (input_ids.shape[1] != 1 or self.training)
                and images is not None):
            vision_tower = vision_tower[0]
            with torch.no_grad():
                if type(images) is list:
                    image_features = []
                    for image in images:
                        out = vision_tower(image.unsqueeze(0), output_hidden_states=True)
                        layer = getattr(self.config, "mm_vision_select_layer", -1)
                        image_features.append(out.hidden_states[layer][:, 1:])
                else:
                    out = vision_tower(images, output_hidden_states=True)
                    layer = getattr(self.config, "mm_vision_select_layer", -1)
                    image_features = out.hidden_states[layer][:, 1:]
            if type(images) is list:
                image_features = [self.mm_projector(f)[0] for f in image_features]
            else:
                image_features = self.mm_projector(image_features)
            dummy = torch.zeros(1, self.config.mm_hidden_size, device=inputs_embeds.device, dtype=inputs_embeds.dtype)
            dummy = self.mm_projector(dummy)
            new_input_embeds = []
            cur_image_idx = 0
            for cur_input_ids, cur_input_embeds in zip(input_ids, inputs_embeds):
                if (cur_input_ids == vision_tower.config.im_patch_token).sum() == 0:
                    cur_input_embeds = cur_input_embeds + (0.0 * dummy).sum()
                    new_input_embeds.append(cur_input_embeds)
                    continue
                if vision_tower.config.use_im_start_end:
                    cur_image_features = image_features[cur_image_idx]
                    num_patches = cur_image_features.shape[0]
                    if (cur_input_ids == vision_tower.config.im_start_token).sum() != (
                            cur_input_ids == vision_tower.config.im_end_token).sum():
                        raise ValueError("Mismatched image start/end token counts.")
                    image_start_tokens = torch.where(
                        cur_input_ids == vision_tower.config.im_start_token)[0]
                    for image_start_token_pos in image_start_tokens:
                        cur_image_features = image_features[cur_image_idx].to(
                            device=cur_input_embeds.device)
                        num_patches = cur_image_features.shape[0]
                        if (cur_input_ids[image_start_token_pos + num_patches + 1]
                                != vision_tower.config.im_end_token):
                            raise ValueError("Image end token should follow image start token.")
                        if orig_embeds_params is not None:
                            cur_new_input_embeds = torch.cat((
                                cur_input_embeds[:image_start_token_pos].detach(),
                                cur_input_embeds[image_start_token_pos: image_start_token_pos + 1],
                                cur_image_features,
                                cur_input_embeds[image_start_token_pos + num_patches + 1:
                                                 image_start_token_pos + num_patches + 2],
                                cur_input_embeds[image_start_token_pos + num_patches + 2:].detach(),
                            ), dim=0)
                        else:
                            cur_new_input_embeds = torch.cat((
                                cur_input_embeds[: image_start_token_pos + 1],
                                cur_image_features,
                                cur_input_embeds[image_start_token_pos + num_patches + 1:],
                            ), dim=0)
                        cur_image_idx += 1
                    new_input_embeds.append(cur_new_input_embeds)
                else:
                    cur_image_features = image_features[cur_image_idx]
                    num_patches = cur_image_features.shape[0]
                    if (cur_input_ids == vision_tower.config.im_patch_token).sum() != num_patches:
                        raise ValueError("Image patch token count mismatch.")
                    masked_indices   = torch.where(
                        cur_input_ids == vision_tower.config.im_patch_token)[0]
                    mask_index_start = masked_indices[0]
                    if (masked_indices != torch.arange(
                            mask_index_start, mask_index_start + num_patches,
                            device=masked_indices.device,
                            dtype=masked_indices.dtype)).any():
                        raise ValueError("Image patch tokens should be consecutive.")
                    if orig_embeds_params is not None:
                        cur_new_input_embeds = torch.cat((
                            cur_input_embeds[:mask_index_start].detach(),
                            cur_image_features,
                            cur_input_embeds[mask_index_start + num_patches:].detach(),
                        ), dim=0)
                    else:
                        cur_new_input_embeds = torch.cat((
                            cur_input_embeds[:mask_index_start],
                            cur_image_features,
                            cur_input_embeds[mask_index_start + num_patches:],
                        ), dim=0)
                    new_input_embeds.append(cur_new_input_embeds)
            inputs_embeds = torch.stack(new_input_embeds, dim=0)
        return super().forward(
            input_ids=None, attention_mask=attention_mask, position_ids=position_ids,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds, use_cache=use_cache,
            output_attentions=output_attentions, output_hidden_states=output_hidden_states,
            return_dict=return_dict, cache_position=cache_position,
        )


class ShikraLlamaForCausalLM(LlamaForCausalLM):
    config_class = ShikraConfig

    def __init__(self, config: ShikraConfig):
        super(LlamaForCausalLM, self).__init__(config)
        self.model   = ShikraLlamaModel(config)
        self.lm_head = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.post_init()

    def forward(self, input_ids=None, attention_mask=None, position_ids=None,
                past_key_values=None, inputs_embeds=None, labels=None, use_cache=None,
                output_attentions=None, output_hidden_states=None, images=None,
                return_dict=None, cache_position=None):
        output_attentions    = output_attentions    or self.config.output_attentions
        output_hidden_states = output_hidden_states or self.config.output_hidden_states
        return_dict          = return_dict if return_dict is not None else self.config.use_return_dict
        outputs = self.model(
            input_ids=input_ids, attention_mask=attention_mask, position_ids=position_ids,
            past_key_values=past_key_values, inputs_embeds=inputs_embeds, use_cache=use_cache,
            output_attentions=output_attentions, output_hidden_states=output_hidden_states,
            return_dict=return_dict, images=images, cache_position=cache_position,
        )
        hidden_states = outputs[0]
        logits        = self.lm_head(hidden_states)
        loss          = None
        if labels is not None:
            shift_logits = logits[..., :-1, :].contiguous()
            shift_labels = labels[..., 1:].contiguous()
            loss_fct     = CrossEntropyLoss()
            shift_logits = shift_logits.view(-1, self.config.vocab_size)
            shift_labels = shift_labels.view(-1).to(shift_logits.device)
            loss = loss_fct(shift_logits, shift_labels)
        if not return_dict:
            output = (logits,) + outputs[1:]
            return (loss,) + output if loss is not None else output
        return CausalLMOutputWithPast(
            loss=loss, logits=logits, past_key_values=outputs.past_key_values,
            hidden_states=outputs.hidden_states, attentions=outputs.attentions,
        )

    def prepare_inputs_for_generation(self, input_ids, past_key_values=None,
                                       attention_mask=None, inputs_embeds=None,
                                       cache_position=None, **kwargs):
        if past_key_values is not None:
            past_length = (past_key_values.get_seq_length()
                           if isinstance(past_key_values, Cache)
                           else past_key_values[0][0].shape[2])
            input_ids = (input_ids[:, past_length:]
                         if input_ids.shape[1] > past_length
                         else input_ids[:, -1:])
        position_ids = kwargs.get("position_ids", None)
        if attention_mask is not None and position_ids is None:
            position_ids = attention_mask.long().cumsum(-1) - 1
            position_ids.masked_fill_(attention_mask == 0, 1)
            if past_key_values is not None:
                position_ids = position_ids[:, -input_ids.shape[1]:]
        model_inputs = ({"inputs_embeds": inputs_embeds}
                        if inputs_embeds is not None and past_key_values is None
                        else {"input_ids": input_ids})
        model_inputs.update({
            "position_ids": position_ids, "cache_position": cache_position,
            "past_key_values": past_key_values, "use_cache": kwargs.get("use_cache"),
            "attention_mask": attention_mask, "images": kwargs.get("images", None),
        })
        return model_inputs

    def initialize_vision_tokenizer(self, mm_use_im_start_end, tokenizer, device,
                                     tune_mm_mlp_adapter=False, pretrain_mm_mlp_adapter=None):
        vision_config = self.model.vision_tower[0].config
        vision_config.use_im_start_end = mm_use_im_start_end
        tokenizer.add_tokens([DEFAULT_IMAGE_PATCH_TOKEN], special_tokens=True)
        self.resize_token_embeddings(len(tokenizer))
        if mm_use_im_start_end:
            num_new_tokens = tokenizer.add_tokens(
                [DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN], special_tokens=True)
            self.resize_token_embeddings(len(tokenizer))
            vision_config.im_start_token, vision_config.im_end_token = (
                tokenizer.convert_tokens_to_ids(
                    [DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN]))
            if num_new_tokens > 0:
                input_embeddings  = self.get_input_embeddings().weight.data
                output_embeddings = self.get_output_embeddings().weight.data
                input_embeddings[-num_new_tokens:]  = input_embeddings[:-num_new_tokens].mean(0, keepdim=True)
                output_embeddings[-num_new_tokens:] = output_embeddings[:-num_new_tokens].mean(0, keepdim=True)
            if tune_mm_mlp_adapter:
                self.model.orig_embeds_params = [
                    self.get_input_embeddings().weight.data.clone().to(device=device)]
                for p in self.get_input_embeddings().parameters():
                    p.requires_grad = True
                for p in self.get_output_embeddings().parameters():
                    p.requires_grad = False
            if pretrain_mm_mlp_adapter:
                mm_projector_weights = torch.load(pretrain_mm_mlp_adapter, map_location="cpu")
                embed_tokens_weight  = mm_projector_weights["model.embed_tokens.weight"]
                input_embeddings     = self.get_input_embeddings().weight.data
                assert num_new_tokens == 2
                if input_embeddings.shape == embed_tokens_weight.shape:
                    input_embeddings[-num_new_tokens:] = embed_tokens_weight[-num_new_tokens:]
                elif embed_tokens_weight.shape[0] == num_new_tokens:
                    input_embeddings[-num_new_tokens:] = embed_tokens_weight
                else:
                    raise ValueError(
                        f"Pretrained: {embed_tokens_weight.shape}. "
                        f"Current: {input_embeddings.shape}. "
                        f"Num new tokens: {num_new_tokens}.")
        vision_config.im_patch_token = tokenizer.convert_tokens_to_ids(
            [DEFAULT_IMAGE_PATCH_TOKEN])[0]


# =============================================================================
# POSTPROCESSING
# =============================================================================

_SP_WORD_BOUNDARY = '\u2581'
_BYTE_FALLBACK    = re.compile(r'^<0x([0-9A-Fa-f]{2})>$')


def decode_output(tokenizer, generated_ids: torch.Tensor) -> str:
    """
    Reconstruct text by reading SentencePiece ▁ markers directly.
    Pure decoding only — NO pathway-specific cleanup here.
    Cleanup is applied by the caller (describe_region / query_region).
    """
    ids         = generated_ids.tolist()
    pieces      = tokenizer.convert_ids_to_tokens(ids)
    special_ids = set(tokenizer.all_special_ids)
    text = ""
    for tok_id, piece in zip(ids, pieces):
        if tok_id in special_ids or piece is None:
            continue
        m = _BYTE_FALLBACK.match(piece)
        if m:
            byte_val = int(m.group(1), 16)
            text += chr(byte_val) if 0x20 <= byte_val <= 0x7E else ' '
            continue
        if piece.startswith(_SP_WORD_BOUNDARY):
            text += ' ' + piece[1:]
        else:
            text += piece
    return re.sub(r' {2,}', ' ', text).strip()


def _strip_vqa_tail(text: str) -> str:
    """
    Strip trailing "The answer is X." only when preceded by real description
    content (≥60 chars). If the ENTIRE output is a PointQA answer, return
    it unchanged so the caller can decide what to do with it.
    """
    m = re.search(r'\s*(So\s+)?[Tt]he answer is[^.]*\.?\s*$', text)
    if m and m.start() >= 60:
        return text[:m.start()].strip()
    return text


def _extract_pointqa_answer(text: str) -> str:
    """
    Extract the payload from "The answer is X." PointQA format.
    Falls back to the full text if the pattern is absent.
    """
    m = re.search(r'[Tt]he answer is\s+([^.]+)\.?', text)
    return m.group(1).strip() if m else text.strip()


# =============================================================================
# MEMORY MANAGEMENT
# =============================================================================

def free_memory(*tensors):
    for t in tensors:
        try:
            del t
        except Exception:
            pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()


def unload_model(model, tokenizer, image_processor):
    print("🧹 Unloading model...")
    try:
        model.cpu()
    except Exception:
        pass
    free_memory(model, tokenizer, image_processor)
    print("✅ Model unloaded, memory freed.")


# =============================================================================
# BBOX + IMAGE UTILITIES
# =============================================================================

def normalize_bbox(bbox_pixels: tuple, image_size: tuple) -> list:
    x1, y1, x2, y2 = bbox_pixels
    w, h = image_size
    clip = lambda v: max(0.0, min(1.0, v))
    return [clip(round(x1/w, 3)), clip(round(y1/h, 3)),
            clip(round(x2/w, 3)), clip(round(y2/h, 3))]


def denormalize_bbox(bbox_norm: list, image_size: tuple) -> tuple:
    x1_n, y1_n, x2_n, y2_n = bbox_norm
    w, h = image_size
    return (int(x1_n*w), int(y1_n*h), int(x2_n*w), int(y2_n*h))


def bbox_to_str(bbox_norm: list) -> str:
    return "[{:.3f},{:.3f},{:.3f},{:.3f}]".format(*bbox_norm)


def _pad_to_square(image: Image.Image) -> Image.Image:
    """
    Pad to square using the mean border colour instead of black.

    Black padding pushes CLIP into out-of-distribution territory because
    natural images never have large solid-black borders. The mean border
    colour is the most neutral background possible — it extends the image
    edge outward in a way that looks like a naturally lighter/darker region.
    """
    w, h = image.size
    if w == h:
        return image

    import numpy as np
    arr  = np.array(image)
    # Mean of the four 1-pixel-wide border strips
    border = np.concatenate([
        arr[0, :, :],   arr[-1, :, :],
        arr[:, 0, :],   arr[:, -1, :],
    ], axis=0)
    fill = tuple(int(v) for v in border.mean(axis=0).round().astype(int))

    size = max(w, h)
    out  = Image.new("RGB", (size, size), fill)
    out.paste(image, ((size - w) // 2, (size - h) // 2))
    return out


def crop_to_bbox(image: Image.Image, bbox_norm: list, padding: float = 0.1) -> Image.Image:
    """
    Crop image to bbox with padding, then pad to square for CLIP.
    padding is clamped to ≥0.1 to prevent PointQA leakage from empty crops.
    """
    padding = max(padding, 0.1)
    w, h = image.size
    x1, y1, x2, y2 = bbox_norm
    pw = (x2 - x1) * padding
    ph = (y2 - y1) * padding
    x1 = max(0.0, x1 - pw);  y1 = max(0.0, y1 - ph)
    x2 = min(1.0, x2 + pw);  y2 = min(1.0, y2 + ph)
    crop = image.crop((int(x1*w), int(y1*h), int(x2*w), int(y2*h))).convert("RGB")
    return _pad_to_square(crop)


def load_image_from_url(url: str, verify_ssl: bool = False) -> Image.Image:
    headers = {"User-Agent": "Mozilla/5.0"}
    resp    = requests.get(url, headers=headers, timeout=15, verify=verify_ssl)
    resp.raise_for_status()
    return Image.open(BytesIO(resp.content)).convert("RGB")


def load_image_from_path(path: str) -> Image.Image:
    return Image.open(path).convert("RGB")


def _load_image(image_url, image_path, verify_ssl):
    if image_url:
        print(f"🖼️  Fetching: {image_url}")
        return load_image_from_url(image_url, verify_ssl=verify_ssl)
    if image_path:
        print(f"🖼️  Loading: {image_path}")
        return load_image_from_path(image_path)
    raise ValueError("Provide image_url or image_path.")


def _resolve_bbox(bbox_pixels, bbox_norm, image):
    if bbox_norm is not None:
        return bbox_norm
    if bbox_pixels is not None:
        # print(f"📏 Source image: {image.size[0]}×{image.size[1]} px")
        return normalize_bbox(bbox_pixels, image.size)
    raise ValueError("Provide bbox_pixels or bbox_norm.")


# =============================================================================
# PROMPT BUILDING
# =============================================================================
# def _build_description_prompt(image_token_len: int = DEFAULT_IMAGE_TOKEN_LEN) -> str:
#     if USE_IM_START_END:
#         image_tokens = (DEFAULT_IM_START_TOKEN
#                         + DEFAULT_IMAGE_PATCH_TOKEN * image_token_len
#                         + DEFAULT_IM_END_TOKEN)
#     else:
#         image_tokens = DEFAULT_IMAGE_PATCH_TOKEN * image_token_len
        
#     return f"{SYSTEM_MSG} USER: {image_tokens}\n{DESCRIPTION_QUESTION} ASSISTANT:"
def _image_tokens(image_token_len: int) -> str:
    if USE_IM_START_END:
        return (
            DEFAULT_IM_START_TOKEN
            + (DEFAULT_IMAGE_PATCH_TOKEN * image_token_len)
            + DEFAULT_IM_END_TOKEN
        )
    return DEFAULT_IMAGE_PATCH_TOKEN * image_token_len


def get_image_token_len_from_vision_tower(model) -> int:
    """
    Compute patch token count from the loaded vision tower.
    Safer than hardcoding 256.
    """
    vc = model.model.vision_tower[0].config
    return (vc.image_size // vc.patch_size) ** 2


def build_gc_prompt(question_template: str, bbox_norm: list, image_token_len: int) -> str:
    """
    Build a Shikra-GC-style prompt:
      - <boxes> -> [x1,y1,x2,y2] normalized
      - <image> -> <im_start><im_patch>*N<im_end>
    """
    q = question_template.replace("<objs>", bbox_to_str(bbox_norm))
    q = q.replace("<boxes>", bbox_to_str(bbox_norm))
    q = q.replace("<image>", _image_tokens(image_token_len))
    return f"{SYSTEM_MSG}\nHuman: {q}\nAssistant:"

def _build_pointqa_prompt(question: str, image_token_len: int = DEFAULT_IMAGE_TOKEN_LEN) -> str:
    if USE_IM_START_END:
        image_tokens = (DEFAULT_IM_START_TOKEN
                        + DEFAULT_IMAGE_PATCH_TOKEN * image_token_len
                        + DEFAULT_IM_END_TOKEN)
    else:
        image_tokens = DEFAULT_IMAGE_PATCH_TOKEN * image_token_len
    return f"{SYSTEM_MSG}\nHuman: {q}\nAssistant:"


# =============================================================================
# MODEL LOADING
# =============================================================================

def _reload_vision_tower(model, tokenizer, device: str):
    tower_name = getattr(model.model.config, "mm_vision_tower", CLIP_MODEL_NAME)
    print(f"🔧 Reloading vision tower: {tower_name}")
    tower = CLIPVisionModel.from_pretrained(tower_name, torch_dtype=torch.float16)
    tower = tower.to(device=device, dtype=torch.float16)
    tower.requires_grad_(False)
    tower.eval()
    tower.config.im_patch_token = tokenizer.convert_tokens_to_ids([DEFAULT_IMAGE_PATCH_TOKEN])[0]
    if USE_IM_START_END:
        im_start_id, im_end_id = tokenizer.convert_tokens_to_ids(
            [DEFAULT_IM_START_TOKEN, DEFAULT_IM_END_TOKEN])
        tower.config.im_start_token   = im_start_id
        tower.config.im_end_token     = im_end_id
        tower.config.use_im_start_end = True
    else:
        tower.config.use_im_start_end = False
    model.model.vision_tower = [tower]
    print(f"✅ Vision tower ready — im_patch_token={tower.config.im_patch_token}")


def load_model(model_path: str, device: str = DEVICE):
    suppress_load_warnings()
    print(f"📦 Loading model from '{model_path}'...")
    tokenizer = LlamaTokenizer.from_pretrained(model_path, use_fast=False)
    model = ShikraLlamaForCausalLM.from_pretrained(
        model_path, torch_dtype=torch.float16, low_cpu_mem_usage=False)
    model = model.to(device)
    model.eval()
    _reload_vision_tower(model, tokenizer, device)
    image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_NAME)
    print(f"✅ Model ready on {device.upper()}")
    return model, tokenizer, image_processor

# import torch
# from transformers import BitsAndBytesConfig

# def load_model(model_path: str, device: str = DEVICE):
#     suppress_load_warnings()
#     print(f"📦 Loading model in 4-bit from '{model_path}'...")
    
#     tokenizer = LlamaTokenizer.from_pretrained(model_path, use_fast=False)

#     # Define the 4-bit configuration
#     bnb_config = BitsAndBytesConfig(
#         load_in_4bit=True,
#         bnb_4bit_compute_dtype=torch.float16,
#         bnb_4bit_quant_type="nf4",  # Normalized Float 4 (usually better than fp4)
#         bnb_4bit_use_double_quant=True,
#     )

#     # Load model with quantization config
#     # Note: device_map="auto" is recommended for 4-bit to handle memory placement
#     model = ShikraLlamaForCausalLM.from_pretrained(
#         model_path, 
#         quantization_config=bnb_config,
#         device_map="auto", # bnb requires device_map for 4-bit loading
#         low_cpu_mem_usage=True
#     )

#     # model.to(device) is often redundant when using device_map="auto"
#     model.eval()
    
#     _reload_vision_tower(model, tokenizer, device)
#     image_processor = CLIPImageProcessor.from_pretrained(CLIP_MODEL_NAME)
    
#     print(f"✅ 4-bit Model ready on {device.upper()}")
#     return model, tokenizer, image_processor


# =============================================================================
# CORE INFERENCE
# =============================================================================

def _run_generate(model, tokenizer, image_tensor, prompt,
                  max_new_tokens, temperature, top_p, device):
    input_ids = tokenizer(
        prompt, return_tensors="pt", truncation=True, max_length=2048,
    ).input_ids.to(device)
    with torch.inference_mode():
        output_ids = model.generate(
            input_ids=input_ids, images=image_tensor,
            do_sample=(temperature > 0), temperature=temperature,
            top_p=top_p, max_new_tokens=max_new_tokens, use_cache=True,
        )
    generated_ids = output_ids[0, input_ids.shape[1]:]
    text          = decode_output(tokenizer, generated_ids)   # pure decode only
    free_memory(input_ids, output_ids, generated_ids)
    return text


# =============================================================================
# PUBLIC API
# =============================================================================

# def describe_region(
#     model, tokenizer, image_processor, image,
#     bbox_pixels: tuple = None, bbox_norm:  list  = None,
#     max_new_tokens: int = MAX_NEW_TOKENS, temperature: float = TEMPERATURE,
#     top_p: float = TOP_P, device: str = DEVICE,
# ) -> dict:
#     """
#     Detailed paragraph description of a bbox region USING the full image context.
#     Uses the optimized prompt to contrast the ROI with its surroundings.
#     No cropping is performed.
#     """
#     bbox_input = _resolve_bbox(bbox_pixels, bbox_norm, image)
    
#     # Fetch the optimal prompt and inject the coordinates
#     question_template = POINTQA_TEMPLATES["describe_roi"]
#     question = question_template.replace("{bbox}", bbox_to_str(bbox_input))

#     # Process the FULL image to preserve spatial context
#     image_tensor = (
#         image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
#         .to(device=device, dtype=torch.float16)
#     )
    
#     # We use _build_pointqa_prompt because the question contains the bbox coordinates
#     raw = _run_generate(model, tokenizer, image_tensor,
#                         _build_pointqa_prompt(question),
#                         max_new_tokens, temperature, top_p, device)
    
#     # Use _strip_vqa_tail (like describe_region) rather than _extract_pointqa_answer 
#     # because we expect a long-form paragraph, not a short "The answer is X" string.
#     description = _strip_vqa_tail(raw)
#     free_memory(image_tensor)
    
#     return {"description": description, "bbox_norm": bbox_input, "question": question}

def describe_region(
    model, tokenizer, image_processor, image,
    bbox_pixels: tuple = None, bbox_norm:  list  = None,
    max_new_tokens: int = MAX_NEW_TOKENS, temperature: float = TEMPERATURE,
    top_p: float = TOP_P, device: str = DEVICE,
) -> dict:
    """
    Detailed ROI caption using Shikra GC-style prompting:
      - question contains <boxes> and <image>
      - <boxes> becomes normalized [x1,y1,x2,y2]
      - <image> becomes <im_start><im_patch>*N<im_end>
      - full image pixels passed via images= (no crop)
    """
    bbox_input = _resolve_bbox(bbox_pixels, bbox_norm, image)

    # Full image (keep consistent with bbox normalization based on original image.size)
    image_tensor = (
        image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
        .to(device=device, dtype=torch.float16)
    )

    # Compute correct patch token length (don’t hardcode 256)
    image_token_len = get_image_token_len_from_vision_tower(model)

    # Build GC prompt
    question = GC_DETAILED_TEMPLATE
    prompt = build_gc_prompt(question, bbox_input, image_token_len)
    # print("PROMPT:", prompt[:300])
    raw = _run_generate(
        model, tokenizer, image_tensor, prompt,
        max_new_tokens, temperature, top_p, device
    )

    # Cleanup: remove image tokens if they leak into decoded text
    description = (raw
                   .replace(DEFAULT_IMAGE_PATCH_TOKEN, "")
                   .replace(DEFAULT_IM_START_TOKEN, "")
                   .replace(DEFAULT_IM_END_TOKEN, "")
                   .strip())
    description = _strip_vqa_tail(description)

    free_memory(image_tensor)

    return {
        "description": description,
        "bbox_norm": bbox_input,
        "question": question,
        "prompt": prompt,
        "raw": raw,
    }

def query_region(
    model, tokenizer, image_processor, image,
    bbox_pixels: tuple = None, bbox_norm:  list  = None,
    query:       str   = "label",
    max_new_tokens: int = MAX_NEW_TOKENS_QA, temperature: float = TEMPERATURE,
    top_p: float = TOP_P, device: str = DEVICE,
) -> dict:
    """
    Keyword / short-answer query about a bbox region via PointQA pathway.
    Full image sent as pixels; bbox coordinates embedded in question text.
    _strip_vqa_tail is NOT applied here — the "The answer is X." format is
    exactly what we want and _extract_pointqa_answer reads it directly.
    """
    bbox_input = _resolve_bbox(bbox_pixels, bbox_norm, image)

    # print(f"📐 Bbox (norm)  : {bbox_to_str(bbox_input)}")
    # if bbox_pixels is not None:
        # print(f"📐 Bbox (pixels): {bbox_pixels}")

    if query in POINTQA_TEMPLATES:
        question_template = POINTQA_TEMPLATES[query]
    else:
        if "{bbox}" not in query:
            raise ValueError(
                f"Custom query must contain {{bbox}} placeholder. Got: '{query}'\n"
                f"Built-in queries: {list(POINTQA_TEMPLATES.keys())}")
        question_template = query

    question = question_template.replace("{bbox}", bbox_to_str(bbox_input))
    # print(f"❓ Query: {question}")

    image_tensor = (
        image_processor.preprocess(image, return_tensors="pt")["pixel_values"]
        .to(device=device, dtype=torch.float16)
    )
    raw    = _run_generate(model, tokenizer, image_tensor,
                           _build_pointqa_prompt(question),
                           max_new_tokens, temperature, top_p, device)
    answer = _extract_pointqa_answer(raw)   # ← reads "The answer is X." directly
    free_memory(image_tensor)

    # print(f"💬 Answer: {answer}")
    return {"answer": answer, "raw": raw, "bbox_norm": bbox_input, "question": question}


# =============================================================================
# ENTRY POINT / QUICK TEST
# =============================================================================

if __name__ == "__main__":
    import pandas as pd
    from tqdm import tqdm
    from torch.utils.data import DataLoader, Subset

    bbox_noise_opts = ['upscale', 'shift']
    sev_opts = [1,3,5]
    for bbox_noise_opt in bbox_noise_opts:
        for sev_opt in sev_opts:
            MODEL_PATH  = "/kaggle/input/models/koahle/shikra-7b/pytorch/default/12/shikra-7b-final"
        
            model, tokenizer, image_processor = load_model(MODEL_PATH)
        
            # ---------------------------------------------------------
            # STEP 1: CONFIGURE BENCHMARK PERTURBATIONS
            # ---------------------------------------------------------
            print("1. Initializing perturbation engine...")
            # NOTE: To evaluate on CLEAN baseline data,
            # set all noise type parameters below to None.
        
            image_noise_type = None
            image_severity   = 0
            bbox_noise_type  = bbox_noise_opt
            bbox_severity    = sev_opt
            mask_noise_type  = None
            mask_severity    = 0
        
            perturber = AddNoise(
                # ==========================================================
                # AXIS 1: FULL-IMAGE CORRUPTIONS (ImageNet-C)
                # ==========================================================
                # Supported: 'gaussian_noise', 'shot_noise', 'impulse_noise',
                #         'defocus_blur', 'glass_blur', 'motion_blur', 'zoom_blur',
                #         'snow', 'frost', 'fog', 'brightness',
                #         'contrast', 'elastic_transform', 'pixelate', 'jpeg_compression'
                image_noise_type=image_noise_type,
                # Severity level: from 1 (mild) to 5 (severe)
                image_severity=image_severity,
        
                # ==========================================================
                # AXIS 2: SPATIAL BBOX PERTURBATIONS (Shift, Upscale, Downscale)
                # ==========================================================
                # Supported:
                # 1. "upscale" (Context expansion)
                # 2. "downscale" (Object truncation)
                # 3. "shift" (Center translation / Misalignment)
                bbox_noise_type=bbox_noise_type,
                # Perturbation severity: from 1 to 5
                bbox_severity=bbox_severity,
        
                # ==========================================================
                # AXIS 3: MASK MORPHOLOGICAL PERTURBATIONS
                # ==========================================================
                # Supported:
                # 1. "dilation" (Boundary dilation)
                # 2. "erosion" (Boundary erosion)
                # 3. "translation" (Mask translation)
                # 4. "boundary_jitter" (Edge contour noise)
                mask_noise_type=mask_noise_type,
                # Mask destruction severity: from 1 to 5
                mask_severity=mask_severity,
            )
        
            dataset_name = "cocostuff" # options: "lvis", "dlc_bench", "cocostuff"
            print("2. Initializing UniversalDataset...")
            if dataset_name == "dlc_bench":
                local_path = '/kaggle/input/datasets/ckhangnguyn/dlc-bench/DLC-Bench'
            else:
                local_path = None
            my_dataset = UniversalDataset(hf_dataset=hf_dataset, dataset_name=dataset_name, perturber=perturber, local_path = local_path)
        
            # ---------------------------------------------------------
            # STEP 2: DATASET CHUNKING (Prevents Kaggle runtime timeouts)
            # ---------------------------------------------------------
            start_index      = 0     # Start sample index
            end_index        = 2000  # End index 
            actual_end_index = min(end_index, len(my_dataset))
        
            subset_indices = list(range(start_index, actual_end_index))
            my_subset      = Subset(my_dataset, subset_indices)
            dataloader     = DataLoader(my_subset, batch_size=1, shuffle=False, collate_fn=lambda x: x[0])
        
            # ---------------------------------------------------------
            # STEP 3: INPUT MODALITY SELECTION
            # ---------------------------------------------------------
            # 'mask': Ground-truth or perturbed mask -> DAM (Upper Bound)
            # 'bbox': Bounding box -> SAM -> Mask -> DAM
            inference_approach = "bbox"
        
            print(f"-> Selected extraction modality: {inference_approach.upper()}")
            print(f"-> Processing images {start_index} to {actual_end_index - 1}. Total: {len(my_subset)} images.")
        
            all_csv_rows = []
        
            # ---------------------------------------------------------
            # STEP 4: MAIN INFERENCE LOOP
            # ---------------------------------------------------------
            print("3. Starting model inference...")
            with torch.no_grad():
                for idx, batch in enumerate(tqdm(dataloader, desc="Processing images")):
                    image        = batch["image"]
                    csv_rows     = batch["csv_rows"]
                    current_img_id = csv_rows[0]["img_id"] if csv_rows else f"Index_{idx}"
        
                    if inference_approach == "mask":
                        regions_to_use = batch["masks"]
                    else:  # "bbox"
                        regions_to_use = batch["bboxes"]
        
                    pred_keywords = []
                    pred_detailed = []
        
                    # FIX 1: enumerate so we have i for the warning message
                    # FIX 2: loop variable renamed to bbox (was mismatched with `region`)
                    for i, bbox in enumerate(regions_to_use):
        
                        # FIX 3: check `bbox is None`, not the undefined `region`
                        if bbox is None:
                            print(
                                f"\n[WARNING] Image '{current_img_id}' - "
                                f"Skipping region #{i + 1} due to missing/corrupted "
                                f"{inference_approach.upper()}!"
                            )
                            pred_keywords.append("")
                            pred_detailed.append("")
                            continue
        
                        # FIX 4: capture return dict in a temp variable so the list
                        #         is not overwritten; extract scalar values from the dict.
        
                        
                        # key_result    = query_region(
                        #     model, tokenizer, image_processor,
                        #     image=image, bbox_pixels=bbox, query="label",
                        # )
                        # pred_keywords.append(key_result["answer"])
                        pred_keywords.append("")
                        
                        detail_result = describe_region(
                            model, tokenizer, image_processor,
                            image=image, bbox_pixels=bbox,
                        )
                        pred_detailed.append(detail_result["description"])
                        # pred_detailed.append("")
        
                    for i, row in enumerate(csv_rows):
                        row["pred_keyword"] = pred_keywords[i] if i < len(pred_keywords) else ""
                        row["pred_detailed"] = pred_detailed[i] if i < len(pred_detailed) else ""
                        all_csv_rows.append(row)
        
                    # CLEANUP PER IMAGE
                    # FIX 7: removed `pred_phrases` from del — it was never defined
                    del regions_to_use, pred_keywords, pred_detailed, image
                    gc.collect()
                    torch.cuda.empty_cache()
        
            # ---------------------------------------------------------
            # STEP 5: EXPORT PREDICTIONS TO CSV FOR BENCHMARK EVALUATORS
            # ---------------------------------------------------------
            print("4. Exporting results to CSV...")
            df = pd.DataFrame(all_csv_rows)
        
            noise_label    = perturber.bbox_noise_type if inference_approach == "bbox" else perturber.mask_noise_type
            severity_label = perturber.bbox_severity   if inference_approach == "bbox" else perturber.mask_severity
            noise_str      = f"{noise_label}_sev{severity_label}" if noise_label else "clean"
        
            csv_filename = (
                f"shikra_{dataset_name}_{inference_approach}_{bbox_noise_type}_{bbox_severity}"
                f"_{start_index}_to_{actual_end_index}.csv"
            )
        
            df.to_csv(csv_filename, index=False, encoding="utf-8-sig")
            print(f"-> Done! Saved {len(df)} region annotations to {csv_filename}")
        
            unload_model(model, tokenizer, image_processor)

## End of Inference Pipeline
